# Factor analysis & predictive feature importance

**Goal.** Identify which characteristics in the sample carry the most predictive signal for communication-apprehension (CA) scores, and summarize the latent factor structure of the PRCA Likert items.

| Analysis | What it answers |
|---|---|
| PRCA item factor analysis (PCA + FA) | How the group / interpersonal items cluster; which items load most strongly |
| Random Forest impurity importance | Which encoded predictors the RF relies on for `gt_group_ca` / `gt_interpersonal_ca` |
| Permutation importance | Which **raw** Prolific/Qualtrics fields most hurt MAE when shuffled |
| Predictor PCA | Dominant directions of variation among persona covariates |

Supporting code: [`src/ca_personas/feature_importance.py`](../src/ca_personas/feature_importance.py).

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ca_personas.feature_importance import (
    likert_item_matrix,
    predictor_feature_importance,
    predictor_pca,
    run_factor_and_importance_bundle,
    run_item_factor_analysis,
)
from ca_personas.load import load_and_prepare, load_qualtrics
from ca_personas.ml_baseline import prepare_modeling_frame

PROLIFIC = ROOT / "data" / "excerpts" / "prolific_excerpt.csv"
QUALTRICS = ROOT / "data" / "excerpts" / "qualtrics_excerpt.csv"
OUT_DIR = ROOT / "outputs" / "feature_importance"
TIER = "transit"  # fullest tabular persona tier for importance

pd.set_option("display.max_colwidth", 80)
print("Project root:", ROOT)

## 2. Load data

- **Likert factor analysis** uses all Qualtrics rows with complete PRCA items (larger N).
- **Predictor importance** uses the Prolific↔Qualtrics inner join (same modeling frame as stage-one ML).

In [ ]:
qualtrics = load_qualtrics(QUALTRICS)
items = likert_item_matrix(qualtrics, scored_direction=True)
participants = load_and_prepare(PROLIFIC, QUALTRICS, how="inner")
model_df = prepare_modeling_frame(participants, tier=TIER)

print(f"Complete PRCA Likert rows for factor analysis: {len(items)}")
print(f"Joined modeling rows ({TIER} tier): {len(model_df)}")
model_df[["participant_id", "Age", "Sex", "Employment status", "gt_group_ca", "gt_interpersonal_ca"]]

## 3. Factor structure of PRCA items

Items are scored so higher values = higher apprehension (comfort items reverse-coded). We report PCA variance explained and FactorAnalysis loadings.

In [ ]:
item_fa = run_item_factor_analysis(items, n_factors=2)
print(f"Using n_factors={item_fa['n_factors']} on n={item_fa['n_rows']} rows")
item_fa["pca_variance"]

In [ ]:
fa_loadings = item_fa["fa_loadings"]
fa_loadings

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
abs_load = fa_loadings.abs()
abs_load.plot(kind="barh", ax=ax)
ax.set_xlabel("|Factor loading|")
ax.set_title("PRCA item |loadings| (FactorAnalysis)")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

item_fa["communalities"]

## 4. Predictive feature importance (persona covariates)

For each CA target we fit a Random Forest on the `transit` tier feature set and report:

1. **Impurity importance** on one-hot/scaled encoded columns  
2. **Permutation importance** on the original survey fields (MAE-based)

::: {.callout-warning}
With a small joined excerpt, importances are unstable. Re-run on the full cohort before strong substantive claims; the ranking API stays the same.
:::

In [ ]:
importances = predictor_feature_importance(
    participants,
    tier=TIER,
    n_repeats=30,
    random_state=42,
)

group_perm = importances["gt_group_ca__raw_permutation"]
inter_perm = importances["gt_interpersonal_ca__raw_permutation"]
print("Top raw features — group CA")
display(group_perm.head(10))
print("Top raw features — interpersonal CA")
display(inter_perm.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False)
for ax, frame, title in [
    (axes[0], group_perm.head(10), "Group CA"),
    (axes[1], inter_perm.head(10), "Interpersonal CA"),
]:
    ax.barh(frame["feature"][::-1], frame["permutation_importance_mean"][::-1], color="#C5050C")
    ax.set_title(title)
    ax.set_xlabel("Permutation importance (↑ MAE when shuffled)")
fig.suptitle("Most important predictive features (raw fields)", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
print("Encoded RF impurity importance — group CA (top 12)")
display(importances["gt_group_ca__encoded_impurity"].head(12))
print("Encoded RF impurity importance — interpersonal CA (top 12)")
display(importances["gt_interpersonal_ca__encoded_impurity"].head(12))

## 5. Predictor-space PCA

PCA on the same encoded design matrix used by stage-one models — useful for seeing which dummy/numeric features dominate multivariate variance (not the same as predictive importance).

In [ ]:
pred_pca = predictor_pca(participants, tier=TIER)
display(pred_pca["variance"])
display(pred_pca["top_loadings"].groupby("component").head(5))

## 6. Persist ranked indicators

Writes CSVs under `outputs/feature_importance/`, including `top_predictive_features.csv` (mean permutation importance across both CA targets).

In [ ]:
paths = run_factor_and_importance_bundle(
    PROLIFIC,
    QUALTRICS,
    OUT_DIR,
    join_how="inner",
    tier=TIER,
)
for name, path in paths.items():
    print(f"{name:40s} -> {path}")

top = pd.read_csv(paths["top_predictive_features"])
top

## 7. How to read these results against LLM / ML baselines

1. Features with high **permutation importance** are the covariates a tabular learner needs most — if an LLM tier that omits them still matches band/score distance, the model is not using the sample’s strongest signals the same way.
2. **Item factor loadings** justify treating group vs interpersonal subscales as distinct evaluation targets (as the persona prompts already do).
3. Re-run this notebook whenever the extract grows; rankings from the tiny excerpt are diagnostic, not definitive.